In [1]:
import numpy as np
import pandas as pd

import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix,
    classification_report,
    roc_curve,
)

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.tree import DecisionTreeClassifier

In [2]:
data = pd.read_csv("archive/events.csv")

In [3]:
data.head()

,id,user_id,sequence_number,session_id,created_at,ip_address,city,state,postal_code,browser,traffic_source,uri,event_type
0,1840721,NaN,3,32c825f3-2e85-4f79-ad2b-188d3ae22785,2022-12-27 02:05:00+00:00,153.11.214.106,São Paulo,São Paulo,02675-031,Chrome,Email,/cancel,cancel
1,1677966,NaN,3,1e21f050-a916-4863-94a0-2a4e474d60c2,2024-06-30 16:04:00+00:00,62.51.52.204,São Paulo,São Paulo,02675-031,Firefox,Adwords,/cancel,cancel
2,1440516,NaN,3,239ccdcb-c9b8-4cab-8203-ef7c4215526f,2024-02-19 07:16:00+00:00,91.225.208.255,São Paulo,São Paulo,02675-031,Chrome,Email,/cancel,cancel
3,1967607,NaN,3,b9082731-7773-4371-b03c-c93d702d7feb,2020-07-17 14:39:00+00:00,90.150.5.159,Aomori City,Aomori,038-0042,Other,Adwords,/cancel,cancel
4,2068472,NaN,3,73feafb9-135d-47eb-8b82-dc73342edf81,2024-09-29 07:56:00+00:00,151.108.206.70,Huanggang,Beijing,100010,Firefox,Adwords,/cancel,cancel


In [4]:
data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 4865324 entries, 0 to 4865323
Data columns (total 13 columns):
 #   Column           Dtype  
---  ------           -----  
 0   id               int64  
 1   user_id          float64
 2   sequence_number  int64  
 3   session_id       object 
 4   created_at       object 
 5   ip_address       object 
 6   city             object 
 7   state            object 
 8   postal_code      object 
 9   browser          object 
 10  traffic_source   object 
 11  uri              object 
 12  event_type       object 
dtypes: float64(1), int64(2), object(10)
memory usage: 482.6+ MB


In [5]:
data.shape

(4865324, 13)

In [6]:
duplicate_columns = []
columns = data.columns

for i in range(len(columns)):
    for j in range(i + 1, len(columns)):
        col_i = columns[i]
        col_j = columns[j]
        if data[col_i].equals(data[col_j]):
            duplicate_columns.append(col_j)

duplicate_columns = list(dict.fromkeys(duplicate_columns))

data = data.drop(columns=duplicate_columns)

print(f"Удалены одинаковые столбцы: {duplicate_columns}")
print(f"Текущий размер data: {data.shape}")

Удалены одинаковые столбцы: []
Текущий размер data: (4865324, 13)


In [7]:
duplicates_count = data.duplicated().sum()
rows_before = len(data)

data = data.drop_duplicates().reset_index(drop=True)

rows_after = len(data)
removed_rows = rows_before - rows_after

print(f"Полных дубликатов найдено: {duplicates_count}")
print(f"Удалено строк: {removed_rows}")
print(f"Размер после очистки: {rows_after}")


Полных дубликатов найдено: 2432662
Удалено строк: 2432662
Размер после очистки: 2432662


In [8]:
fe = []
for col in data.columns:
    if (len(data[col].unique()) == data.shape[0]): fe.append(col)
data = data.drop(columns=fe)
print(len(fe))

1


In [9]:
data.shape

(2432662, 12)